In [6]:
!pip install llama-index wikipedia rank_bm25 sentence-transformers chromadb groq llama-index-llms-groq


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# API key is now loaded from .env file
from llama_index.llms.groq import Groq
from llama_index.core import Settings

Settings.llm = Groq(model="llama-3.1-8b-instant")

In [4]:
import wikipedia
from llama_index.core import Document

def load_wikipedia_docs(query_list):
    docs = []
    for q in query_list:
        try:
            page = wikipedia.page(q)
            docs.append(Document(text=page.content, metadata={"title": page.title}))
        except:
            pass
    return docs

In [5]:
def generate_queries(question):
    prompt = f"""
    Generate 5 search topics related to this research question.
    Only return topics, one per line.

    Question: {question}
    """
    response = Settings.llm.complete(prompt)
    return response.text.strip().split("\n")

In [6]:
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex

def build_index(documents):
    splitter = SentenceSplitter(chunk_size=300, chunk_overlap=50)
    nodes = splitter.get_nodes_from_documents(documents)
    
    embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
    index = VectorStoreIndex(nodes, embed_model=embed_model)
    
    return index, nodes

In [7]:
from llama_index.retrievers.bm25 import BM25Retriever

def bm25(nodes):
    return BM25Retriever.from_defaults(nodes=nodes)

In [8]:
def hybrid_search(query, index, bm25_retriever, top_k=5):
    vector_retriever = index.as_retriever(similarity_top_k=top_k)
    
    vector_results = vector_retriever.retrieve(query)
    bm25_results = bm25_retriever.retrieve(query)
    
    results = {}
    
    for r in vector_results:
        results[r.node.node_id] = r.node.text
        
    for r in bm25_results:
        results[r.node.node_id] = r.node.text
        
    return list(results.values())

In [9]:
def research_assistant(question):
    print("Generating queries...")
    queries = generate_queries(question)
    print("Topics:", queries)
    
    print("Loading Wikipedia documents...")
    docs = load_wikipedia_docs(queries)
    
    print("Building index...")
    index, nodes = build_index(docs)
    
    bm25_retriever = bm25(nodes)
    
    print("Retrieving context...")
    context = []
    for q in queries:
        results = hybrid_search(q, index, bm25_retriever)
        context.extend(results)
    
    context_text = "\n\n".join(context[:5])
    
    prompt = f"""
    Use the context below from multiple documents to answer the question.

    Context:
    {context_text}

    Question:
    {question}
    """
    
    response = Settings.llm.complete(prompt)
    
    print("\nFinal Answer:\n")
    print(response.text)

In [8]:
research_assistant("How does Retrieval Augmented Generation reduce hallucinations?")
research_assistant("What is hybrid search in information retrieval?")
research_assistant("Compare RAG and fine-tuning")

Generating queries...


AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

In [10]:
# Test API key
try:
    test_response = Settings.llm.complete("Say 'Hello, API key is working!'")
    print("✅ API key is valid!")
    print("Test response:", test_response.text)
except Exception as e:
    print("❌ API key error:", str(e))
    print("Please check your GROQ_API_KEY in the cell above")

research_assistant("How does Retrieval Augmented Generation reduce hallucinations?")
research_assistant("What is hybrid search in information retrieval?")
research_assistant("Compare RAG and fine-tuning")

✅ API key is valid!
Test response: Hello, API key is working!
Generating queries...
Topics: ['1. Effects of Retrieval Augmented Generation on Textual Hallucinations', '2. Reducing Hallucinations in Language Models using Retrieval-Augmented Generation', '3. A Comparative Study of Retrieval-Augmented Generation and Traditional Generation Methods on Hallucination Rates', '4. Investigating the Role of Retrieval-Augmented Generation in Mitigating Hallucinations in Conversational AI', '5. Evaluating the Impact of Retrieval-Augmented Generation on Hallucination Reduction in Text Generation Tasks']
Loading Wikipedia documents...
Building index...
Retrieving context...

Final Answer:

The provided context does not explicitly mention how Retrieval Augmented Generation (RAG) reduces hallucinations. However, based on general knowledge about RAG, it is a technique that combines the strengths of both generative models (like GPT-3) and retrieval-based models.

Retrieval Augmented Generation typically